# MedoraAI Chest X-ray Production Training - EfficientNet-B4

This notebook trains the backend-compatible chest X-ray classifier using the full NIH ChestX-ray14 dataset, not the old 20K-image smoke-test subset.

Backend compatibility target:

```python
timm.create_model("efficientnet_b4", pretrained=True, num_classes=15)
```

Exported artifact:

```text
models/chest_xray_efficientnet_b4.pt
```

Production policy in this notebook:

- Use all available NIH ChestX-ray14 images by default.
- Split by `Patient ID` to reduce patient leakage.
- Train with mixed precision, weighted BCE, best-validation-AUC checkpointing, and early stopping.
- Save validation/test metrics, thresholds, label manifest, and an export zip.
- Do not blindly run infinite epochs; let validation AUC decide when to stop.

Hardware sizing guide:

| GPU tier | Practical use | Starting batch size | Suggested schedule |
|---|---:|---:|---:|
| 16 GB VRAM | Feasible but slower | 16-32 | head 3 epochs, fine 20-30, gradient accumulation 2 |
| 24 GB VRAM, e.g. RTX 4090 | Good single-GPU baseline | 48-80 | head 3, fine 30-45 |
| 48 GB VRAM, e.g. L40S/A6000-class | Strong rented GPU choice | 96-160 | head 3, fine 35-60 |
| 80 GB VRAM, e.g. A100/H100 | Best single-GPU throughput | 160-256 | head 3, fine 40-80 |

For production accuracy, prefer more clean validation and external test data over simply increasing epochs. The NIH labels are weak image-level labels; for localization-sensitive pneumothorax Grad-CAM, plan a second fine-tune/evaluation pass on SIIM-ACR or another segmentation-annotated pneumothorax dataset.

In [ ]:
# Runtime setup
!nvidia-smi || true

# Do not force reinstall torch on managed GPU images unless needed.
!python -m pip install -q --upgrade timm pandas scikit-learn tqdm pillow matplotlib seaborn kaggle


In [ ]:
# Configuration
from pathlib import Path
import os

WORKSPACE = Path(os.environ.get('WORKSPACE', '/workspace'))
FAST_DATA_ROOT = Path(os.environ.get('FAST_DATA_ROOT', '/root/nih_chest_xray14_fast'))
WORKSPACE_DATA_ROOT = WORKSPACE / 'nih_chest_xray14'
TMP_ROOT = WORKSPACE / 'tmp'
CACHE_ROOT = WORKSPACE / '.cache'
KAGGLE_DIR = WORKSPACE / '.kaggle'

for path in [WORKSPACE, FAST_DATA_ROOT, WORKSPACE_DATA_ROOT, TMP_ROOT, CACHE_ROOT, KAGGLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['TMPDIR'] = str(TMP_ROOT)
os.environ['XDG_CACHE_HOME'] = str(CACHE_ROOT)
os.environ['KAGGLE_CONFIG_DIR'] = str(KAGGLE_DIR)

# Full dataset by default. Set an integer only for a quick smoke test.
DEBUG_MAX_IMAGES = None

# Training schedule. Early stopping usually decides the actual epoch count.
EPOCHS_HEAD = 3
EPOCHS_FINE = 50
EARLY_STOP_PATIENCE = 8
MIN_FINE_EPOCHS = 12

# Tune these based on VRAM. If OOM, halve BATCH_SIZE first.
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '96'))
GRAD_ACCUM_STEPS = int(os.environ.get('GRAD_ACCUM_STEPS', '1'))
IMG_SIZE = 224
RESIZE_SIZE = 256

SEED = 42
VAL_SIZE = 0.10
TEST_SIZE = 0.10

OUT_DIR = WORKSPACE / 'medoraai_chest_production'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / 'chest_xray_efficientnet_b4.pt'
BEST_CKPT_PATH = OUT_DIR / 'best_checkpoint.pt'
LABELS_PATH = OUT_DIR / 'chest_xray_efficientnet_b4.labels.json'
THRESHOLDS_PATH = OUT_DIR / 'chest_xray_thresholds.json'
METRICS_PATH = OUT_DIR / 'metrics.json'
LOG_CSV_PATH = OUT_DIR / 'training_log.csv'
EXPORT_ZIP = WORKSPACE / 'medoraai_chest_xray_model_export.zip'

print('WORKSPACE:', WORKSPACE)
print('BATCH_SIZE:', BATCH_SIZE, 'GRAD_ACCUM_STEPS:', GRAD_ACCUM_STEPS)
print('OUT_DIR:', OUT_DIR)


In [ ]:
# Dataset discovery / optional Kaggle download
import getpass
import json
import subprocess

EXPECTED_PNGS = 112120
MIN_READY_PNGS = 100000

def find_csv(root: Path):
    return list(root.rglob('Data_Entry_2017.csv'))

def count_pngs(root: Path) -> int:
    return sum(1 for _ in root.rglob('*.png'))

def dataset_ready(root: Path) -> bool:
    csvs = find_csv(root)
    pngs = count_pngs(root) if csvs else 0
    print(f'{root}: csv={len(csvs)} pngs={pngs}')
    return bool(csvs) and pngs >= MIN_READY_PNGS

print('Disk space:')
subprocess.run(['df', '-h', str(WORKSPACE)], check=False)
subprocess.run(['df', '-h', '/'], check=False)

if dataset_ready(FAST_DATA_ROOT):
    DATA_ROOT = FAST_DATA_ROOT
elif dataset_ready(WORKSPACE_DATA_ROOT):
    DATA_ROOT = WORKSPACE_DATA_ROOT
else:
    DATA_ROOT = FAST_DATA_ROOT
    kaggle_json = KAGGLE_DIR / 'kaggle.json'
    if not kaggle_json.exists():
        username = input('Kaggle username: ').strip()
        api_key = getpass.getpass('Kaggle API key: ').strip()
        kaggle_json.write_text(json.dumps({'username': username, 'key': api_key}), encoding='utf-8')
        kaggle_json.chmod(0o600)
    print('Downloading NIH ChestX-ray14 via Kaggle. This can take a while.')
    subprocess.run([
        'kaggle', 'datasets', 'download', '-d', 'nih-chest-xrays/data',
        '-p', str(DATA_ROOT), '--unzip'
    ], check=True)

csv_candidates = find_csv(DATA_ROOT)
if not csv_candidates:
    raise FileNotFoundError(f'Data_Entry_2017.csv not found under {DATA_ROOT}')
CSV_PATH = csv_candidates[0]
print('Using DATA_ROOT:', DATA_ROOT)
print('Using CSV:', CSV_PATH)


In [ ]:
# Imports and deterministic-ish setup
import csv
import math
import random
import shutil
import time
import zipfile
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
from PIL import Image, ImageFile
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

CLASS_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural Thickening', 'Hernia', 'No Finding'
]

def autocast_context():
    if USE_AMP:
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()


In [ ]:
# Metadata table and labels
df = pd.read_csv(CSV_PATH)
print(df.head())
print('Raw rows:', len(df))

print('Indexing image paths...')
image_paths = {p.name: str(p) for p in DATA_ROOT.rglob('*.png')}
df['path'] = df['Image Index'].map(image_paths)
df = df.dropna(subset=['path']).reset_index(drop=True)
print('Rows with images:', len(df))

if DEBUG_MAX_IMAGES is not None and len(df) > DEBUG_MAX_IMAGES:
    df = df.sample(DEBUG_MAX_IMAGES, random_state=SEED).reset_index(drop=True)
    print('DEBUG sampled rows:', len(df))

def encode_labels(label_text: str) -> list[float]:
    labels = set(str(label_text).split('|'))
    return [1.0 if label in labels else 0.0 for label in CLASS_LABELS]

label_matrix = np.array([encode_labels(x) for x in df['Finding Labels']], dtype=np.float32)
for i, label in enumerate(CLASS_LABELS):
    df[label] = label_matrix[:, i]

print(pd.Series(label_matrix.sum(axis=0), index=CLASS_LABELS).sort_values(ascending=False))
print('Unique patients:', df['Patient ID'].nunique())


In [ ]:
# Patient-level train/val/test split
groups = df['Patient ID'].values
indices = np.arange(len(df))

first_split = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE + TEST_SIZE, random_state=SEED)
train_idx, temp_idx = next(first_split.split(indices, groups=groups))

temp_groups = groups[temp_idx]
relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
second_split = GroupShuffleSplit(n_splits=1, test_size=relative_test_size, random_state=SEED + 1)
val_rel_idx, test_rel_idx = next(second_split.split(temp_idx, groups=temp_groups))
val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

def split_report(name, idx):
    y = label_matrix[idx]
    return {
        'name': name,
        'images': int(len(idx)),
        'patients': int(df.iloc[idx]['Patient ID'].nunique()),
        'positives': {label: int(y[:, i].sum()) for i, label in enumerate(CLASS_LABELS)},
    }

split_reports = [split_report('train', train_idx), split_report('val', val_idx), split_report('test', test_idx)]
for report in split_reports:
    print(report['name'], 'images=', report['images'], 'patients=', report['patients'])
    print(pd.Series(report['positives']).sort_values(ascending=False).head(8))

(OUT_DIR / 'split_report.json').write_text(json.dumps(split_reports, indent=2), encoding='utf-8')


In [ ]:
# Dataset and dataloaders
class ChestDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, indices: np.ndarray, transform):
        self.frame = frame.iloc[indices].reset_index(drop=True)
        self.transform = transform
        self.targets = self.frame[CLASS_LABELS].values.astype(np.float32)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        image = self.transform(image)
        target = torch.from_numpy(self.targets[idx])
        return image, target

train_transform = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomRotation(degrees=7, fill=0),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05), fill=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ChestDataset(df, train_idx, train_transform)
val_ds = ChestDataset(df, val_idx, eval_transform)
test_ds = ChestDataset(df, test_idx, eval_transform)

NUM_WORKERS = min(8, os.cpu_count() or 2)
loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda',
    persistent_workers=NUM_WORKERS > 0,
)
train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

print('Batches:', len(train_loader), len(val_loader), len(test_loader), 'workers:', NUM_WORKERS)


In [ ]:
# Model, loss, and optimizer helpers
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=len(CLASS_LABELS))
model = model.to(DEVICE)
if DEVICE.type == 'cuda':
    model = model.to(memory_format=torch.channels_last)

train_targets = label_matrix[train_idx]
pos = train_targets.sum(axis=0)
neg = len(train_targets) - pos
pos_weight_np = np.clip(neg / np.maximum(pos, 1.0), 1.0, 50.0).astype(np.float32)
pos_weight = torch.tensor(pos_weight_np, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print('pos_weight:')
print(pd.Series(pos_weight_np, index=CLASS_LABELS).round(2))

def set_trainable_head_only():
    for p in model.parameters():
        p.requires_grad = False
    classifier = model.get_classifier()
    for p in classifier.parameters():
        p.requires_grad = True

def set_trainable_all():
    for p in model.parameters():
        p.requires_grad = True

def make_optimizer(lr: float, weight_decay: float = 1e-4):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)

def make_scaler():
    return torch.amp.GradScaler('cuda', enabled=USE_AMP)


In [ ]:
# Metrics
def multilabel_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(np.float32)
    per_label = {}
    aucs = []
    for i, label in enumerate(CLASS_LABELS):
        label_true = y_true[:, i]
        label_prob = y_prob[:, i]
        label_pred = y_pred[:, i]
        item = {
            'support': int(label_true.sum()),
            'precision': float(precision_score(label_true, label_pred, zero_division=0)),
            'recall': float(recall_score(label_true, label_pred, zero_division=0)),
            'f1': float(f1_score(label_true, label_pred, zero_division=0)),
        }
        if len(np.unique(label_true)) > 1:
            item['auc'] = float(roc_auc_score(label_true, label_prob))
            aucs.append(item['auc'])
        else:
            item['auc'] = None
        per_label[label] = item
    return {
        'macro_auc': float(np.mean(aucs)) if aucs else None,
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'per_label': per_label,
    }

def find_best_thresholds(y_true: np.ndarray, y_prob: np.ndarray) -> dict[str, float]:
    thresholds = {}
    grid = np.linspace(0.05, 0.95, 91)
    for i, label in enumerate(CLASS_LABELS):
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            score = f1_score(y_true[:, i], (y_prob[:, i] >= t).astype(np.float32), zero_division=0)
            if score > best_f1:
                best_t, best_f1 = float(t), float(score)
        thresholds[label] = round(best_t, 4)
    return thresholds


In [ ]:
# Train/evaluate loops
def run_epoch(loader, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    total_loss = 0.0
    all_targets, all_probs = [], []
    if train:
        optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, leave=False)
    for step, (images, targets) in enumerate(pbar, start=1):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        if DEVICE.type == 'cuda':
            images = images.to(memory_format=torch.channels_last)

        with torch.set_grad_enabled(train):
            with autocast_context():
                logits = model(images)
                loss = criterion(logits, targets)
                loss_for_backward = loss / max(GRAD_ACCUM_STEPS, 1)

            if train:
                scaler.scale(loss_for_backward).backward()
                if step % GRAD_ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.detach().cpu()) * images.size(0)
        probs = torch.sigmoid(logits.detach()).cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets.detach().cpu().numpy())
        pbar.set_postfix(loss=float(loss.detach().cpu()))

    if train and len(loader) % GRAD_ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    y_true = np.concatenate(all_targets, axis=0)
    y_prob = np.concatenate(all_probs, axis=0)
    avg_loss = total_loss / max(len(loader.dataset), 1)
    metrics = multilabel_metrics(y_true, y_prob)
    return avg_loss, metrics, y_true, y_prob

def append_log(row: dict):
    exists = LOG_CSV_PATH.exists()
    with LOG_CSV_PATH.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)

best_auc = -1.0
best_epoch_name = None

def save_best_if_needed(phase: str, epoch: int, val_loss: float, val_metrics: dict):
    global best_auc, best_epoch_name
    val_auc = val_metrics['macro_auc'] or -1.0
    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch_name = f'{phase}_{epoch}'
        torch.save(model.state_dict(), OUT_PATH)
        torch.save({
            'phase': phase,
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
            'val_metrics': val_metrics,
            'class_labels': CLASS_LABELS,
        }, BEST_CKPT_PATH)
        print('Saved best model:', OUT_PATH, 'macro_auc=', best_auc)
        return True
    return False


In [ ]:
# Training phases
def fit_phase(phase: str, epochs: int, lr: float, min_epochs: int = 0):
    optimizer = make_optimizer(lr=lr)
    scaler = make_scaler()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )
    bad_epochs = 0
    phase_best = -1.0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_metrics, _, _ = run_epoch(train_loader, optimizer=optimizer, scaler=scaler)
        val_loss, val_metrics, _, _ = run_epoch(val_loader)
        val_auc = val_metrics['macro_auc'] or -1.0
        scheduler.step(val_auc)
        improved_global = save_best_if_needed(phase, epoch, val_loss, val_metrics)

        if val_auc > phase_best + 1e-4:
            phase_best = val_auc
            bad_epochs = 0
        else:
            bad_epochs += 1

        row = {
            'phase': phase,
            'epoch': epoch,
            'lr': optimizer.param_groups[0]['lr'],
            'train_loss': train_loss,
            'train_macro_auc': train_metrics['macro_auc'],
            'train_micro_f1': train_metrics['micro_f1'],
            'val_loss': val_loss,
            'val_macro_auc': val_metrics['macro_auc'],
            'val_micro_f1': val_metrics['micro_f1'],
            'seconds': round(time.time() - t0, 2),
            'saved_best': improved_global,
        }
        append_log(row)
        print(row)

        if epoch >= min_epochs and bad_epochs >= EARLY_STOP_PATIENCE:
            print(f'Early stopping {phase}: no val AUC improvement for {bad_epochs} epochs.')
            break

set_trainable_head_only()
print('Phase 1: train classifier head')
fit_phase('head', EPOCHS_HEAD, lr=1e-3, min_epochs=EPOCHS_HEAD)

set_trainable_all()
print('Phase 2: fine-tune full network')
fit_phase('fine', EPOCHS_FINE, lr=1e-4, min_epochs=MIN_FINE_EPOCHS)

print('Best:', best_epoch_name, best_auc)


In [ ]:
# Final validation/test evaluation from the best checkpoint
state = torch.load(OUT_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.eval()

val_loss, val_metrics, val_true, val_prob = run_epoch(val_loader)
test_loss, test_metrics, test_true, test_prob = run_epoch(test_loader)
thresholds = find_best_thresholds(val_true, val_prob)

metrics_report = {
    'best_epoch': best_epoch_name,
    'best_val_macro_auc_during_training': best_auc,
    'val_loss': val_loss,
    'val_metrics_at_0_5': val_metrics,
    'test_loss': test_loss,
    'test_metrics_at_0_5': test_metrics,
    'thresholds_from_val_best_f1': thresholds,
    'config': {
        'batch_size': BATCH_SIZE,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'epochs_head': EPOCHS_HEAD,
        'epochs_fine_max': EPOCHS_FINE,
        'patient_level_split': True,
        'debug_max_images': DEBUG_MAX_IMAGES,
    },
}
METRICS_PATH.write_text(json.dumps(metrics_report, indent=2), encoding='utf-8')
THRESHOLDS_PATH.write_text(json.dumps(thresholds, indent=2), encoding='utf-8')
print(json.dumps({
    'val_macro_auc': val_metrics['macro_auc'],
    'test_macro_auc': test_metrics['macro_auc'],
    'test_micro_f1_at_0_5': test_metrics['micro_f1'],
    'pneumothorax_test': test_metrics['per_label']['Pneumothorax'],
}, indent=2))


In [ ]:
# Plots for quick review
log_df = pd.read_csv(LOG_CSV_PATH)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=log_df, x='epoch', y='val_macro_auc', hue='phase', marker='o', ax=axes[0])
axes[0].set_title('Validation macro AUC')
sns.lineplot(data=log_df, x='epoch', y='val_loss', hue='phase', marker='o', ax=axes[1])
axes[1].set_title('Validation loss')
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=160)
plt.show()

per_label_auc = {
    label: item['auc'] for label, item in test_metrics['per_label'].items() if item['auc'] is not None
}
plt.figure(figsize=(10, 5))
pd.Series(per_label_auc).sort_values().plot(kind='barh')
plt.title('Test AUC by label')
plt.tight_layout()
plt.savefig(OUT_DIR / 'test_auc_by_label.png', dpi=160)
plt.show()


In [ ]:
# Backend-compatible manifest and export zip
manifest = {
    'backend_expected_model': 'timm efficientnet_b4 num_classes=15',
    'labels': CLASS_LABELS,
    'state_dict_path': './models/chest_xray_efficientnet_b4.pt',
    'training_dataset': 'NIH ChestX-ray14',
    'split': 'patient-level train/val/test split',
    'notes': 'Image-level weak labels; not pixel-level pneumothorax localization supervision.',
}
LABELS_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

with zipfile.ZipFile(EXPORT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT_PATH, 'models/chest_xray_efficientnet_b4.pt')
    zf.write(LABELS_PATH, 'models/chest_xray_efficientnet_b4.labels.json')
    zf.write(METRICS_PATH, 'models/chest_xray_training_metrics.json')
    zf.write(THRESHOLDS_PATH, 'models/chest_xray_thresholds.json')
    zf.write(OUT_DIR / 'split_report.json', 'models/chest_xray_split_report.json')
    if LOG_CSV_PATH.exists():
        zf.write(LOG_CSV_PATH, 'models/chest_xray_training_log.csv')

print('Model:', OUT_PATH)
print('Labels:', LABELS_PATH)
print('Metrics:', METRICS_PATH)
print('Export zip:', EXPORT_ZIP)
print('Copy/export zip into the MedoraAI repo root, then extract so backend uses models/chest_xray_efficientnet_b4.pt')


## After Training

1. Download `/workspace/medoraai_chest_xray_model_export.zip`.
2. Put it in the MedoraAI repo root.
3. Extract it so these files update:

```text
models/chest_xray_efficientnet_b4.pt
models/chest_xray_efficientnet_b4.labels.json
```

4. Run full evaluation locally:

```powershell
python tools/evaluate_chest_model.py --data-root C:\path\to\nih_chest_xray14 --output-json chest_eval.json
python tools/audit_chest_gradcam.py --labels-csv C:\path\to\labels.csv --data-root C:\path\to\images
```

Important: better classification training can improve the gradient signal, but NIH image-level labels alone are still weak localization supervision. For pneumothorax localization quality, add SIIM-ACR pneumothorax masks or another segmentation/bbox dataset in a second-stage training/evaluation notebook.